In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Load Data
cols = ['id', 'entity', 'sentiment', 'text']
train_df = pd.read_csv('data/twitter_training.csv', names=cols).dropna(subset=['text'])
val_df = pd.read_csv('data/twitter_validation.csv', names=cols).dropna(subset=['text'])

# 2. Text Preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

train_df['clean_text'] = train_df['text'].apply(clean_text)
val_df['clean_text'] = val_df['text'].apply(clean_text)

# 3. Vectorization (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = vectorizer.fit_transform(train_df['clean_text'])
X_val = vectorizer.transform(val_df['clean_text'])

y_train = train_df['sentiment']
y_val = val_df['sentiment']

# 4. Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_val)

print(f"Accuracy: {accuracy_score(y_val, y_pred):.2f}")
print(classification_report(y_val, y_pred))

# 5. K-Means Clustering (Example on subset)
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X_train[:10000])
train_subset = train_df.iloc[:10000].copy()
train_subset['cluster'] = kmeans.labels_

# Analyze cluster vs sentiment
cluster_analysis = train_subset.groupby(['cluster', 'sentiment']).size().unstack()
print(cluster_analysis)

Accuracy: 0.80
              precision    recall  f1-score   support

  Irrelevant       0.78      0.70      0.74       172
    Negative       0.77      0.87      0.82       266
     Neutral       0.84      0.75      0.79       285
    Positive       0.81      0.85      0.83       277

    accuracy                           0.80      1000
   macro avg       0.80      0.79      0.80      1000
weighted avg       0.80      0.80      0.80      1000

sentiment  Irrelevant  Negative  Neutral  Positive
cluster                                           
0                  41       159      105       217
1                1682      1416     1217      2129
2                  76       310      389       678
3                  64       421      890       206
